# InvoiceRL: LoRA SFT + GRPO with verifiable rewards for invoice extraction

Fine-tune a small open model (**Qwen2.5-0.5B-Instruct**) to extract invoice fields as JSON:

1. **Synthetic data**: generate invoices with exact ground-truth JSON (held-out test split).
2. **Baseline**: evaluate the base model zero-shot.
3. **LoRA SFT**: supervised fine-tuning with low-rank adapters.
4. **GRPO**: reinforcement learning where a *program* scores each output (JSON validity, field match, and an arithmetic check on line items, subtotal, tax and total).
5. **Evaluate before and after every stage** and report results.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU**. Total time is roughly 1.5–2.5 hours (GRPO is the slow step).

In [ ]:
# If install fails, remove the version pins and re-run.
!pip install -q "transformers==4.53.2" "trl==0.19.1" "peft==0.16.0" "accelerate>=1.4" datasets faker

In [ ]:
import json, random, re, time, gc
import torch
from faker import Faker
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
SEED = 42
N_TRAIN, N_TEST = 1500, 200
EVAL_N = 200          # test examples used for each evaluation
random.seed(SEED); torch.manual_seed(SEED)
fake = Faker("en_IN"); Faker.seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Synthetic invoices with exact ground truth

In [ ]:
ITEMS = ["Solar panel 400W", "Inverter 5kW", "Copper cable 10m", "Smart meter", "Battery pack 5kWh",
         "Mounting kit", "Installation service", "Annual maintenance", "LED fixture", "Power strip"]
TAX_RATES = [0.05, 0.12, 0.18]

def make_invoice():
    items = []
    for _ in range(random.randint(1, 4)):
        items.append({"description": random.choice(ITEMS),
                      "qty": random.randint(1, 20),
                      "price": round(random.uniform(50, 25000), 2)})
    subtotal = round(sum(i["qty"] * i["price"] for i in items), 2)
    rate = random.choice(TAX_RATES)
    tax = round(subtotal * rate, 2)
    total = round(subtotal + tax, 2)
    gold = {"vendor": fake.company(), "invoice_no": f"INV-{random.randint(10000, 99999)}",
            "date": fake.date_between("-2y", "today").isoformat(),
            "line_items": items, "subtotal": subtotal, "tax": tax, "total": total}
    return gold, rate

def render(gold, rate):
    """Render the same data in varied, slightly messy layouts so the task isn't trivial."""
    style = random.randint(0, 2)
    d = gold["date"]
    if random.random() < 0.5:                       # vary the date format
        y, m, dd = d.split("-"); d = f"{dd}/{m}/{y}"
    lines = []
    if style == 0:
        lines += [gold["vendor"].upper(), f"Invoice #: {gold['invoice_no']}", f"Date: {d}", "", "Item | Qty | Unit price"]
        lines += [f"{i['description']} | {i['qty']} | {i['price']:.2f}" for i in gold["line_items"]]
        lines += ["", f"Subtotal: {gold['subtotal']:.2f}", f"GST @ {int(rate*100)}%: {gold['tax']:.2f}", f"TOTAL DUE: {gold['total']:.2f}"]
    elif style == 1:
        lines += [f"Bill from {gold['vendor']}", f"Ref {gold['invoice_no']} issued on {d}"]
        lines += [f"- {i['qty']} x {i['description']} @ Rs {i['price']:,.2f}" for i in gold["line_items"]]
        lines += [f"Net amount Rs {gold['subtotal']:,.2f}; tax Rs {gold['tax']:,.2f}; grand total Rs {gold['total']:,.2f}"]
    else:
        lines += [f"TAX INVOICE  {gold['invoice_no']}", f"Seller: {gold['vendor']}   Dated {d}", "Qty  Description  Rate"]
        lines += [f"{i['qty']}  {i['description']}  {i['price']}" for i in gold["line_items"]]
        lines += [f"Sub-total {gold['subtotal']}", f"Tax {gold['tax']}", f"Amount payable {gold['total']}"]
    return "\n".join(lines)

SYSTEM = ("Extract the invoice into JSON with keys: vendor, invoice_no, date (YYYY-MM-DD), "
          "line_items (list of {description, qty, price}), subtotal, tax, total. Numbers must be plain numbers. "
          "Return only the JSON.")

def make_split(n):
    rows = []
    for _ in range(n):
        gold, rate = make_invoice()
        rows.append({"invoice": render(gold, rate), "answer": json.dumps(gold)})
    return rows

train_rows, test_rows = make_split(N_TRAIN), make_split(N_TEST)
print(train_rows[0]["invoice"], "\n---\n", train_rows[0]["answer"])

## 2. Model, prompts and evaluation

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
if tok.pad_token is None: tok.pad_token = tok.eos_token

def to_prompt(invoice):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": invoice}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def load(path=MODEL_ID):
    return AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map="cuda")

In [ ]:
FIELDS = ["vendor", "invoice_no", "date", "subtotal", "tax", "total"]

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m: return None
    try: return json.loads(m.group(0))
    except Exception: return None

def num_eq(a, b):
    try: return abs(float(a) - float(b)) < 0.01
    except Exception: return False

def field_ok(pred, gold, f):
    return num_eq(pred.get(f), gold[f]) if f in ("subtotal", "tax", "total") else str(pred.get(f, "")).strip() == str(gold[f])

def arithmetic_ok(pred):
    """Calculator check: line items add up to the subtotal, and subtotal + tax = total."""
    try:
        s = sum(float(i["qty"]) * float(i["price"]) for i in pred["line_items"])
        return abs(s - float(pred["subtotal"])) < 0.05 and abs(float(pred["subtotal"]) + float(pred["tax"]) - float(pred["total"])) < 0.05
    except Exception:
        return False

def score(text, gold):
    pred = parse_json(text)
    if not isinstance(pred, dict):
        return {"json_valid": 0, "field_acc": 0.0, "all_fields": 0, "arithmetic": 0}
    ok = [field_ok(pred, gold, f) for f in FIELDS]
    return {"json_valid": 1, "field_acc": sum(ok) / len(ok), "all_fields": int(all(ok)), "arithmetic": int(arithmetic_ok(pred))}

@torch.no_grad()
def evaluate(model, rows, bs=16, label=""):
    model.eval(); res = []
    for i in range(0, len(rows), bs):
        batch = rows[i:i+bs]
        enc = tok([to_prompt(r["invoice"]) for r in batch], return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=320, do_sample=False, pad_token_id=tok.pad_token_id)
        texts = tok.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        res += [score(t, json.loads(r["answer"])) for t, r in zip(texts, batch)]
    agg = {k: round(100 * sum(r[k] for r in res) / len(res), 1) for k in res[0]}
    print(label, agg); return agg

RESULTS = {}

## 3. Baseline (zero-shot)

In [ ]:
model = load()
RESULTS["Base (zero-shot)"] = evaluate(model, test_rows[:EVAL_N], label="BASE")
del model; gc.collect(); torch.cuda.empty_cache()

## 4. LoRA supervised fine-tuning (SFT)

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

sft_ds = Dataset.from_list([{"messages": [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": r["invoice"]},
    {"role": "assistant", "content": r["answer"]}]} for r in train_rows])

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])

sft_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, device_map="cuda")
sft_args = SFTConfig(output_dir="sft-out", num_train_epochs=1, per_device_train_batch_size=8,
                     gradient_accumulation_steps=2, learning_rate=2e-4, lr_scheduler_type="cosine",
                     warmup_ratio=0.05, logging_steps=10, max_length=768, fp16=True,
                     save_strategy="no", report_to="none", seed=SEED)
t0 = time.time()
trainer = SFTTrainer(model=sft_model, args=sft_args, train_dataset=sft_ds, peft_config=lora, processing_class=tok)
trainer.train()
print(f"SFT took {(time.time()-t0)/60:.1f} min")

merged = trainer.model.merge_and_unload()
merged.save_pretrained("sft-merged"); tok.save_pretrained("sft-merged")
del trainer, sft_model, merged; gc.collect(); torch.cuda.empty_cache()

model = load("sft-merged")
RESULTS["LoRA SFT"] = evaluate(model, test_rows[:EVAL_N], label="SFT")
del model; gc.collect(); torch.cuda.empty_cache()

## 5. GRPO with a verifiable reward

The reward comes from a **program, not a judge model**: 0.2 for valid JSON, 0.1 for each correct field (up to 0.6), and 0.2 when the line items, subtotal, tax and total are arithmetically consistent. The maximum is 1.0.

In [ ]:
from trl import GRPOTrainer, GRPOConfig

def invoice_reward(completions, answer, **kwargs):
    rewards = []
    for text, gold_s in zip(completions, answer):
        gold, pred = json.loads(gold_s), parse_json(text)
        if not isinstance(pred, dict):
            rewards.append(0.0); continue
        r = 0.2 + 0.1 * sum(field_ok(pred, gold, f) for f in FIELDS)
        r += 0.2 if arithmetic_ok(pred) else 0.0
        rewards.append(round(r, 3))
    return rewards

grpo_ds = Dataset.from_list([{"prompt": to_prompt(r["invoice"]), "answer": r["answer"]} for r in train_rows[:600]])

grpo_model = AutoModelForCausalLM.from_pretrained("sft-merged", torch_dtype=torch.float32, device_map="cuda")
grpo_args = GRPOConfig(output_dir="grpo-out", learning_rate=1e-5, per_device_train_batch_size=8,
                       gradient_accumulation_steps=1, num_generations=4, max_prompt_length=512,
                       max_completion_length=320, max_steps=150, logging_steps=10, fp16=True,
                       save_strategy="no", report_to="none", seed=SEED)
t0 = time.time()
gtrainer = GRPOTrainer(model=grpo_model, reward_funcs=[invoice_reward], args=grpo_args,
                       train_dataset=grpo_ds, processing_class=tok,
                       peft_config=LoraConfig(r=16, lora_alpha=32, task_type="CAUSAL_LM",
                                              target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
gtrainer.train()
print(f"GRPO took {(time.time()-t0)/60:.1f} min")

grpo_merged = gtrainer.model.merge_and_unload()
grpo_merged.save_pretrained("grpo-merged"); tok.save_pretrained("grpo-merged")
del gtrainer, grpo_model, grpo_merged; gc.collect(); torch.cuda.empty_cache()

model = load("grpo-merged")
RESULTS["LoRA SFT + GRPO"] = evaluate(model, test_rows[:EVAL_N], label="GRPO")

## 6. Results

In [ ]:
hdr = "| Model | JSON valid % | Field accuracy % | All fields correct % | Arithmetic consistent % |"
sep = "|---|---|---|---|---|"
rows = [f"| {k} | {v['json_valid']} | {v['field_acc']} | {v['all_fields']} | {v['arithmetic']} |" for k, v in RESULTS.items()]
table = "\n".join([hdr, sep] + rows)
print(table)
with open("results.md", "w") as f:
    f.write(f"Held-out test set: {EVAL_N} synthetic invoices (seed {SEED}).\n\n" + table + "\n")
json.dump(RESULTS, open("results.json", "w"), indent=2)

## 7. Error analysis (fill in after running)
Look at a few test cases where the final model is still wrong and note the patterns in the README. For example: date-format confusion, commas in amounts, missed line items, or rounding.

```python
# Quick look at failures
```

In [ ]:
fails = []
with torch.no_grad():
    for r in test_rows[:60]:
        enc = tok([to_prompt(r["invoice"])], return_tensors="pt").to(model.device)
        out = model.generate(**enc, max_new_tokens=320, do_sample=False, pad_token_id=tok.pad_token_id)
        txt = tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        if score(txt, json.loads(r["answer"]))["all_fields"] == 0:
            fails.append((r["invoice"], txt, r["answer"]))
print(len(fails), "failures out of 60")
for inv, pred, gold in fails[:3]:
    print("INVOICE:\n", inv, "\nPRED:", pred, "\nGOLD:", gold, "\n" + "-"*60)